In [1]:
# Compatibility aliases for any stale JSON-style booleans
false = False
true = True

from pathlib import Path
import importlib.util
import shutil
import time
import traceback
from collections import Counter

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RUN_ROOT = PROJECT_ROOT
SRC = PROJECT_ROOT / "src" / "training_v2.py"

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
TRAIN_T1 = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

if not SRC.exists():
    raise FileNotFoundError(f"Training module not found: {SRC}")
if not TRAIN_T1.exists() or not TRAIN_MASKS.exists():
    raise FileNotFoundError(f"Missing training subfolders under {TRAIN_DIR}")

spec = importlib.util.spec_from_file_location("seg", SRC)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load module spec from {SRC}")
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

METHOD_NAME = 'Sliding-Window + Gaussian Blending'

# Common baseline settings (kept aligned across experiments)
INPUT_SHAPE = (112, 112, 96, 1)
PATCH_SIZE = (112, 112, 96)
PATCHES_PER_CASE = 2
EPOCH_STEPS = 60
FIT_VERBOSE = 2
MEMORY_LOGS_ENABLED = False
TOTAL_EPOCHS = 30
INITIAL_EPOCH = 0

BASE_FILTERS = 6
SAM_HEADS = 2
BATCH_SIZE = 1
VAL_SPLIT = 1.0 / 3.0
DROPOUT_RATE = 0.55
L2_REG = 0.0015

AUG_INTENSITY = 0.45
ROTATION_RANGE = 25
SMALL_LESION_THRESHOLD = 6000
SYNTHETIC_LESION_PROB = 0.6

INITIAL_LR = 1e-4
MIN_LR = 5e-7
WARMUP_EPOCHS = 15
COSINE_FIRST_CYCLE_EPOCHS = 40
COSINE_T_MUL = 1.0
COSINE_M_MUL = 1.0
SWA_EPOCHS = 0
SWA_LR_MULT = None

DICE_WEIGHT = 0.4
BOUNDARY_WEIGHT = 0.6
BOUNDARY_WARMUP_DICE = 0.4
BOUNDARY_WARMUP_BOUNDARY = 0.6
BOUNDARY_RAMP_EPOCHS = 1

FOCAL_TVERSKY_WEIGHT = 0.0
TVERSKY_ALPHA = 0.7
TVERSKY_BETA = 0.3
FOCAL_TVERSKY_GAMMA = 1.5

SIZE_BUCKET_PROBS = (0.35, 0.25, 0.20, 0.12, 0.08)
PATCH_FG_PROB_BY_BIN = (0.95, 0.90, 0.80, 0.65, 0.55)

LOAD_FULL_IMAGE_FOR_PATCHING = True
FULL_RES_TARGET_SHAPE = None
WHOLE_BRAIN_VAL_ENABLED = True
WHOLE_BRAIN_VAL_EVERY_N_EPOCHS = 1
WHOLE_BRAIN_VAL_MAX_CASES = None
WHOLE_BRAIN_VAL_TTA = False
PATCH_SAMPLING_STRATEGY = "hemisphere"
HEMISPHERE_AXIS = 2
HEMISPHERE_BALANCED = True

EXTRA_OVERRIDES = {
    "GAUSSIAN_TILE_OVERLAP": 0.75,
    "GAUSSIAN_TILE_SIGMA": 0.18,
    "WHOLE_BRAIN_VAL_TTA": False
}
for k, v in EXTRA_OVERRIDES.items():
    globals()[k] = v

# Preview split composition so val has representative cases by source
preview_model_dir = RUN_ROOT / "_preview_models"
preview_callbacks_dir = RUN_ROOT / "_preview_callbacks"
preview_model_dir.mkdir(parents=True, exist_ok=True)
preview_callbacks_dir.mkdir(parents=True, exist_ok=True)
preview_cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    INPUT_SHAPE=INPUT_SHAPE,
    PATCH_SIZE=PATCH_SIZE,
    BATCH_SIZE=BATCH_SIZE,
    VALIDATION_SPLIT=VAL_SPLIT,
    MODEL_DIR=preview_model_dir,
    CALLBACKS_DIR=preview_callbacks_dir,
)
_pairs, _lesion = seg.load_generic_dataset(preview_cfg)
_train_pairs, _val_pairs = seg.create_stratified_splits(_pairs, _lesion, batch_size=BATCH_SIZE, test_size=VAL_SPLIT)

def _src_name(pair):
    name = Path(str(pair[0])).name
    return name.split("__", 1)[0] if "__" in name else name.split("_", 1)[0]

print("Method:", METHOD_NAME)
print("Train composition:", dict(Counter(_src_name(p) for p in _train_pairs)))
print("Val composition  :", dict(Counter(_src_name(p) for p in _val_pairs)))

shutil.rmtree(preview_model_dir, ignore_errors=True)
shutil.rmtree(preview_callbacks_dir, ignore_errors=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
for d in (MODEL_DIR, CALLBACKS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Using training module:", SRC)
print("Training data:", TRAIN_DIR)
print("Run dir:", RUN_DIR)

train_kwargs = dict(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
    INPUT_SHAPE=INPUT_SHAPE,
    BASE_FILTERS=BASE_FILTERS,
    SAM_HEADS=SAM_HEADS,
    BATCH_SIZE=BATCH_SIZE,
    DROPOUT_RATE=DROPOUT_RATE,
    L2_REG=L2_REG,
    PATCH_SIZE=PATCH_SIZE,
    PATCHES_PER_CASE=PATCHES_PER_CASE,
    EPOCH_STEPS=EPOCH_STEPS,
    FIT_VERBOSE=FIT_VERBOSE,
    MEMORY_LOGS_ENABLED=MEMORY_LOGS_ENABLED,
    TOTAL_EPOCHS=TOTAL_EPOCHS,
    INITIAL_EPOCH=INITIAL_EPOCH,
    RESAMPLE_TO_TARGET=False,
    AUGMENTATION_INTENSITY=AUG_INTENSITY,
    ROTATION_RANGE=ROTATION_RANGE,
    SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
    SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
    INITIAL_LR=INITIAL_LR,
    MIN_LR=MIN_LR,
    WARMUP_EPOCHS=WARMUP_EPOCHS,
    COSINE_FIRST_CYCLE_EPOCHS=COSINE_FIRST_CYCLE_EPOCHS,
    COSINE_T_MUL=COSINE_T_MUL,
    COSINE_M_MUL=COSINE_M_MUL,
    COSINE_MIN_LR_MULT=0.1,
    SWA_EPOCHS=SWA_EPOCHS,
    SWA_LR_MULT=SWA_LR_MULT,
    DICE_WEIGHT=DICE_WEIGHT,
    BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
    DICE_LOSS_WEIGHT=0.4,
    BOUNDARY_LOSS_WEIGHT=0.6,
    BOUNDARY_WARMUP_DICE=BOUNDARY_WARMUP_DICE,
    BOUNDARY_WARMUP_BOUNDARY=BOUNDARY_WARMUP_BOUNDARY,
    BOUNDARY_RAMP_EPOCHS=BOUNDARY_RAMP_EPOCHS,
    FOCAL_TVERSKY_WEIGHT=FOCAL_TVERSKY_WEIGHT,
    TVERSKY_ALPHA=TVERSKY_ALPHA,
    TVERSKY_BETA=TVERSKY_BETA,
    FOCAL_TVERSKY_GAMMA=FOCAL_TVERSKY_GAMMA,
    SIZE_BUCKET_PROBS=SIZE_BUCKET_PROBS,
    PATCH_FG_PROB_BY_BIN=PATCH_FG_PROB_BY_BIN,
    LOAD_FULL_IMAGE_FOR_PATCHING=LOAD_FULL_IMAGE_FOR_PATCHING,
    FULL_RES_TARGET_SHAPE=FULL_RES_TARGET_SHAPE,
    WHOLE_BRAIN_VAL_ENABLED=WHOLE_BRAIN_VAL_ENABLED,
    WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=WHOLE_BRAIN_VAL_EVERY_N_EPOCHS,
    WHOLE_BRAIN_VAL_MAX_CASES=WHOLE_BRAIN_VAL_MAX_CASES,
    WHOLE_BRAIN_VAL_TTA=WHOLE_BRAIN_VAL_TTA,
    PATCH_SAMPLING_STRATEGY=PATCH_SAMPLING_STRATEGY,
    HEMISPHERE_AXIS=HEMISPHERE_AXIS,
    HEMISPHERE_BALANCED=HEMISPHERE_BALANCED,
    DIFF_AWARE_ENABLED=True,
    DIFF_EMA_LAMBDA=0.8,
    DIFF_BETA=1.5,
    VALIDATION_SPLIT=VAL_SPLIT,
    LOAD_WEIGHTS_FROM=None,
    RESUME_FROM_LATEST=False,
)
train_kwargs.update(EXTRA_OVERRIDES)

try:
    history = seg.train_dynamic_model(**train_kwargs)
    print("Training complete. Keys:", list(getattr(history, "history", {}).keys()))
    print("Artifacts saved to", RUN_DIR)
except Exception:
    traceback.print_exc()
    raise

latest_link = RUN_ROOT / "runs" / "latest"
if latest_link.exists() or latest_link.is_symlink():
    latest_link.unlink()
latest_link.symlink_to(RUN_DIR, target_is_directory=True)

best_src = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if best_src.exists():
    best_copy = RUN_ROOT / "runs" / "latest_best.weights.h5"
    shutil.copy2(best_src, best_copy)
    print("Saved best copy ->", best_copy)


2026-03-12 13:44:00.939551: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1773344643.393277 2157724 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1773344643.395142 2157724 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1773344643.395495 2157724 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1773344643.397335 2157724 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-03-12 13:44:03,492 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-03-12 13:44:03,493 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-03-12 13:44:03,493 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy


2026-03-12 13:44:05,045 - SmartSOTA_Dynamic - INFO - Manifest composition: {'Approx-Numeracy-Processed': 3, 'ATLAS-Images-f0d7431e': 3, 'ARC-combined-t1-raw-ab0d1794': 3}
2026-03-12 13:44:05,046 - SmartSOTA_Dynamic - INFO - 📊 Created 9 image–mask pairs from manifest
2026-03-12 13:44:05,046 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2026-03-12 13:44:05,047 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=1.04GB | GPU mem tracking failed | Disk: 600.7GB free
2026-03-12 13:44:05,056 - SmartSOTA_Dynamic - INFO - 🧮 Dataset split (stratified_source+lesion): Train=6 (66.7%), Validation=3 (33.3%)
2026-03-12 13:44:05,056 - SmartSOTA_Dynamic - INFO - 🧩 Stratification groups: {'ARC-combined-t1-raw-ab0d1794|lesion=1': 3, 'ATLAS-Images-f0d7431e|lesion=1': 3, 'Approx-Numeracy-Processed|lesion=1': 3}
2026-03-12 13:44:05,057 - SmartSOTA_Dynamic - INFO - ⚖️ Lesion prevalence: Train=100.00%, Validation=100.00%
2026-03-12 13:44:05,060 - SmartSOTA_Dynamic - INFO - 🔧 Config: smart_

...

Epoch 30/30


2026-03-12 14:59:34,271 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 3/3 cases
2026-03-12 14:59:34,272 - SmartSOTA_Dynamic - INFO - Whole-brain val @epoch 29: soft_macro=0.04970 soft_micro=0.05057 hard_macro@thr0.50=0.00001 (cases=3, 141.8s)
2026-03-12 14:59:34,273 - SmartSOTA_Dynamic - INFO - Whole-brain val by source (soft_macro): {'ARC-combined-t1-raw-ab0d1794': 0.014456286231376745, 'ATLAS-Images-f0d7431e': 0.09218749545077883, 'Approx-Numeracy-Processed': 0.04246652996914729}
2026-03-12 14:59:34,273 - SmartSOTA_Dynamic - INFO - Whole-brain val by source (hard_macro@thr0.50): {'ARC-combined-t1-raw-ab0d1794': 6.508721686637026e-11, 'ATLAS-Images-f0d7431e': 9.071529006131705e-12, 'Approx-Numeracy-Processed': 4.406064945267755e-05}



Epoch 30: val_dice_coefficient improved from 0.04958 to 0.04970, saving model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/ARC_ATLAS_Train_Experiment/01_sliding_window_gaussian/runs/20260312_134405/callbacks/best_model_dynamic.weights.h5
60/60 - 150s - 3s/step - dice_coefficient: 0.0857 - loss: 0.8663 - safe_binary_iou: 0.0428 - val_dice_coefficient: 0.0497 - val_whole_dice_micro: 0.0506 - val_whole_dice_hard: 1.4687e-05


2026-03-12 14:59:35,058 - SmartSOTA_Dynamic - INFO - Training complete: dict_keys(['dice_coefficient', 'loss', 'safe_binary_iou', 'val_dice_coefficient', 'val_whole_dice_micro', 'val_whole_dice_hard'])


Training complete. Keys: ['dice_coefficient', 'loss', 'safe_binary_iou', 'val_dice_coefficient', 'val_whole_dice_micro', 'val_whole_dice_hard']
Artifacts saved to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/ARC_ATLAS_Train_Experiment/01_sliding_window_gaussian/runs/20260312_134405
Saved best copy -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/ARC_ATLAS_Train_Experiment/01_sliding_window_gaussian/runs/latest_best.weights.h5


In [1]:
# Quick sanity prediction on zeros (standalone-safe)
from pathlib import Path
import importlib.util
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC = PROJECT_ROOT / "src" / "training_v2.py"

if "seg" not in globals():
    if not SRC.exists():
        raise FileNotFoundError(f"Training module not found: {SRC}")
    spec = importlib.util.spec_from_file_location("seg", SRC)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load module spec from {SRC}")
    seg = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(seg)

# Fallback defaults if cell 1 wasn't run in this kernel
TRAIN_DIR = globals().get("TRAIN_DIR", PROJECT_ROOT / "data" / "train")
TRAIN_T1 = globals().get("TRAIN_T1", TRAIN_DIR / "t1")
TRAIN_MASKS = globals().get("TRAIN_MASKS", TRAIN_DIR / "masks")
INPUT_SHAPE = globals().get("INPUT_SHAPE", (112, 112, 96, 1))
PATCH_SIZE = globals().get("PATCH_SIZE", (112, 112, 96))
BASE_FILTERS = globals().get("BASE_FILTERS", 6)
SAM_HEADS = globals().get("SAM_HEADS", 2)

# Prefer active run from cell 1, else use runs/latest symlink
RUN_DIR = globals().get("RUN_DIR", None)
if RUN_DIR is None:
    latest_link = PROJECT_ROOT / "runs" / "latest"
    if latest_link.exists():
        RUN_DIR = latest_link.resolve()
    else:
        run_root = PROJECT_ROOT / "runs"
        run_dirs = sorted([p for p in run_root.glob("20*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
        if not run_dirs:
            raise FileNotFoundError("No run directory found under runs/. Run training cell first or set RUN_DIR.")
        RUN_DIR = run_dirs[-1]

MODEL_DIR = globals().get("MODEL_DIR", RUN_DIR / "models")
CALLBACKS_DIR = globals().get("CALLBACKS_DIR", RUN_DIR / "callbacks")

cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    INPUT_SHAPE=INPUT_SHAPE,
    BASE_FILTERS=BASE_FILTERS,
    SAM_HEADS=SAM_HEADS,
    PATCH_SIZE=PATCH_SIZE,
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
)

weights = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if weights.exists():
    print("Loading weights:", weights)
    m = seg.build_model_for_inference(cfg, weights_path=str(weights))
else:
    print("No best weights found at", weights, "- using randomly initialized model.")
    m = seg.build_model_for_inference(cfg)

x0 = np.zeros((1, *INPUT_SHAPE), np.float32)
p0 = m.predict(x0, verbose=0)[0, ..., 0]
print("Blank input -> p.mean=", float(p0.mean()), " p.max=", float(p0.max()))


2026-03-12 15:59:13.889325: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1773352756.761289 3421187 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1773352756.763223 3421187 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1773352756.766696 3421187 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1773352756.768522 3421187 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-03-12 15:59:16,858 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-03-12 15:59:16,859 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-03-12 15:59:16,859 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Loading weights: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/ARC_ATLAS_Train_Experiment/01_sliding_window_gaussian/runs/20260312_134405/callbacks/best_model_dynamic.weights.h5


2026-03-12 15:59:19.057532: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
2026-03-12 15:59:19.057666: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
2026-03-12 15:59:19.057695: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]
2026-03-12 15:59:21.067397: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001


Blank input -> p.mean= 0.001984622096642852  p.max= 0.06475082039833069
